# Analysis of Gromacs-run oligonucleotide MD simulations
This is a quick guide to setting up a linear workflow for automated analysis and feature extraction for Molecular Dynamics simulations of oligonucleotides.

## Imports
We begin with all top-level requirements and imports, and ensure our software dependencies are setup correctly.

In [ ]:
from maize.core.workflow import Workflow
from maize.utilities.io import Config
from pathlib import Path
from maize.steps.mai.gromacs_rna.features_extract import OligoAnalysis

## Workflow

Maize will look for configurations of the specific softwares, in particular the locations of the software packages requested. These configuration should be contained in a TOML file. Note that you will need to set up the required software yourself. 

In [ ]:
flow = Workflow(name="oligo_analysis", cleanup_temp=False, level="debug")

flow.config = Config()
flow.config.update(Path("oligoanalysis_config.toml"))

Create the node `OligoAnalysis`. This node takes simulations data from Gromacs runs as input, performs analysis using tools: Gromacs, ProLIF, MDAnalysis, x3dna, and generates a csv file containing the list of systems with their corresponding MD features from analysis.


In [ ]:
analysis = flow.add(OligoAnalysis, name="oligo_analysis")

Here we set up inputs to the Node. 

`input_dir`is where you enter the path to the input data that contains the simulations data from the Gromacs simulations run on Maize through the `MDsRNA` node.
`send_csv` : A csv file containing the list of sequences and chemical modifications thereof. For more detailed description of the csv file, look at `pnab_oligo_wf.ipynb` notebook. 
`output`: Path wherein to save the output from this run.

Note that this node does not have an output port.

In [ ]:
inp_dir = Path(
    "../maize/steps/mai/gromacs_rna/data/md_output"
)

send_csv = Path(
    "../maize/steps/mai/gromacs_rna/data/sequence_data.csv"
)

output = Path(
    "/path/output/"
)

### Parameters

Here set up parameters for the node. In this example, we set input and parameters to run analysis for simulations of 2 oligo systems with 1 replicate each. Consequently, we enter '3' for `molecules` and '2' for `replicas`. You can decide to skip some number of frames (`skip_frames`) in the trajectory for analysis. The number (nr) you enter means you skip every nr-th frame. If you do not want to skip frames, set this parameter to '1'. For x3dna analysis, the trajectory is converted to a pdb file for which you can also skip frames through `skip_frames_pdb`. For ProLIF analysis, you need to specify the number of CPU threads to use. We choose '1' here. For ProLIF analysis of the interface between the guide and target strands of a duplex, you need to specify the cutoff distance through `cutoff_interface`. We choose 10 Å here. Similarly, for analyzing contacts of the oligo with ions, you need to specify the cutoff distance, which is chosen as 4.5 Å here. 


In [ ]:
analysis.inp_data_path.set(inp_dir)
analysis.inp_csv.set(send_csv)
analysis.dest_path.set(output)
analysis.molecules.set(2)
analysis.replicas.set(1)
analysis.skip_frames.set(1)
analysis.skip_frames_pdb.set(1)
analysis.num_jobs.set(1)
analysis.cutoff_interface.set(10)
analysis.cutoff_ions.set(4.5)

### Check
If this method doesn't throw an exception, we have connected everything correctly and set all required parameters.

In [ ]:
flow.check()

### Run
Run the workflow

In [ ]:
flow.execute()